# Notebook 05: Message Personalization

**Purpose:** Generate personalized outreach email messages for each researcher

**Input:** `data/researchers_enriched.csv`

**Output:** `data/researchers_outreach.csv`

This notebook uses LLM to create tailored email messages based on each researcher's profile, interests, and recent work.

## Setup

In [ ]:
# Import required libraries
import pandas as pd
import time
from tqdm import tqdm

# Add project root to path for imports
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

# Import our utilities
from scripts.utils import load_config, get_openai_client, call_llm, load_prompt_template, ensure_data_dir, save_csv_checkpoint, load_csv_checkpoint

# Load configuration
config = load_config()
print(f"Loaded config for domain: {config['domain']['name']}")

# Set up data directory
data_dir = ensure_data_dir(config['output']['data_dir'])

# Initialize LLM client
client = get_openai_client(config)
print("LLM client initialized")

## Load Enriched Researcher Data

In [ ]:
# Load enriched researchers from previous step
researchers_df = load_csv_checkpoint("researchers_enriched.csv")
if researchers_df is None:
    raise FileNotFoundError("researchers_enriched.csv not found. Please run notebook 04 first.")

print(f"Loaded {len(researchers_df)} enriched researchers")
researchers_df.head()

## Message Personalization Functions

In [ ]:
def format_researcher_context(row):
    """Format researcher information for the LLM prompt"""
    name = row['name']
    affiliation = row['affiliation'] if row['affiliation'] != 'Unknown' else 'independent researcher'
    research_focus = row['research_focus']
    seniority = row['seniority']
    
    # Format social profiles
    profiles = []
    if row.get('linkedin'):
        profiles.append(f"LinkedIn: {row['linkedin']}")
    if row.get('twitter'):
        profiles.append(f"Twitter: {row['twitter']}")
    if row.get('github'):
        profiles.append(f"GitHub: {row['github']}")
    
    profiles_text = "\n".join(profiles) if profiles else "No public social media profiles found"
    
    # Get recent paper info (using first paper as example)
    papers = row['papers'].split(';')[:1]  # Just use first paper for context
    recent_work = f"Recent paper: {papers[0]}" if papers[0] else ""
    
    context = f"""Researcher: {name}
Affiliation: {affiliation}
Seniority Level: {seniority}
Research Focus: {research_focus}
{recent_work}
Social Profiles:
{profiles_text}
"""
    
    return context

def generate_personalized_message(row, email_config):
    """Generate a personalized email message for a researcher"""
    
    # Format researcher context
    researcher_context = format_researcher_context(row)
    
    # Load the personalization prompt template
    prompt = load_prompt_template("personalize_message")
    
    # Fill in template variables
    prompt = prompt.format(
        researcher_info=researcher_context,
        tone=email_config['tone'],
        background=email_config['background'],
        value_proposition=email_config['value_proposition'],
        length=email_config['length'],
        call_to_action=email_config['call_to_action']
    )
    
    # Generate message using LLM
    message = call_llm(
        client=client,
        prompt=prompt,
        model=config['llm']['model'],
        temperature=config['llm']['temperature'],
        max_tokens=config['llm']['max_tokens']
    )
    
    return message.strip()

# Test the message generation
print("Testing message personalization...")
test_researcher = researchers_df.iloc[0]
test_context = format_researcher_context(test_researcher)
print("Sample researcher context:")
print(test_context[:300] + "...")

## Generate Personalized Messages

In [ ]:
# Get email configuration
email_config = config['email_template']
print(f"Email template config: tone='{email_config['tone']}', length='{email_config['length']}'")

# Set processing limits
max_researchers = config['processing'].get('max_researchers', len(researchers_df))
batch_size = config['processing'].get('batch_size', 5)

# Limit to configured max
researchers_to_process = researchers_df.head(max_researchers).copy()
print(f"Generating messages for {len(researchers_to_process)} researchers (batch size: {batch_size})")

# Initialize results
results = []

# Process in batches
for i in tqdm(range(0, len(researchers_to_process), batch_size), desc="Processing batches"):
    batch = researchers_to_process.iloc[i:i+batch_size]
    
    # Generate messages for batch
    batch_with_messages = batch.copy()
    messages = []
    
    for idx, row in batch.iterrows():
        try:
            message = generate_personalized_message(row, email_config)
            messages.append(message)
            print(f"Generated message for {row['name']} ({len(message)} chars)")
        except Exception as e:
            print(f"Failed to generate message for {row['name']}: {e}")
            messages.append("Error generating message")
    
    # Add message column
    batch_with_messages['personalized_message'] = messages
    batch_with_messages['status'] = 'pending'
    batch_with_messages['sent_date'] = ''
    batch_with_messages['notes'] = ''
    
    results.append(batch_with_messages)
    
    # Save intermediate checkpoint every 3 batches
    if (i // batch_size + 1) % 3 == 0:
        intermediate_df = pd.concat(results, ignore_index=True)
        save_csv_checkpoint(intermediate_df, "researchers_outreach_checkpoint.csv")
        print(f"\nSaved intermediate checkpoint after {len(intermediate_df)} researchers")

# Combine all results
outreach_df = pd.concat(results, ignore_index=True)

print(f"\nCompleted processing {len(outreach_df)} researchers")
print(f"Generated {len(outreach_df)} personalized messages")

## Review and Save Results

In [ ]:
# Show sample messages
print("Sample personalized messages:")
for i in range(min(3, len(outreach_df))):
    row = outreach_df.iloc[i]
    print(f"\n--- Message for {row['name']} ---")
    print(row['personalized_message'][:300] + "..." if len(row['personalized_message']) > 300 else row['personalized_message'])
    print(f"Status: {row['status']}")

In [ ]:
# Save final results
save_csv_checkpoint(outreach_df, "researchers_outreach.csv")
print(f"\nSaved {len(outreach_df)} outreach-ready researchers to data/researchers_outreach.csv")

# Summary statistics
total_messages = len(outreach_df)
pending_messages = (outreach_df['status'] == 'pending').sum()
avg_message_length = outreach_df['personalized_message'].str.len().mean()

print("\nMessage Personalization Summary:")
print(f"Total messages generated: {total_messages}")
print(f"Messages ready to send: {pending_messages}")
print(f"Average message length: {avg_message_length:.0f} characters")
print(f"Messages with errors: {total_messages - pending_messages}")

## Next Steps

The outreach data is now ready for email sending in **scripts/send_emails.py**.